<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 5


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

**Описание задачи:**  
Создать базовый класс ***Book*** в C#, который будет представлять информацию о
книгах. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.
     
**Требования к базовому классу Book:**  
* Атрибуты: Название (***Title***), Автор (***Author***), Год издания (***YearOfPublication***).  
* Методы:  
    * ***GetInfo()***: метод для получения информации о книге в виде строки.  
    * ***Read()***: метод для вывода сообщения о чтении книги.  
    * ***Borrow()***: метод для вывода сообщения о выдаче книги на чтение.  
  
**Требования к производным классам:**
* Учебник (***Textbook***): Должен содержать дополнительные атрибуты, такие как
Предмет (***Subject***). Метод *Read()* должен быть переопределен для
добавления информации о предмете при чтении учебника.
* Художественная литература (***Fiction***): Должен содержать дополнительные
атрибуты, такие как Жанр (***Genre***). Метод *Borrow()* должен быть
переопределен для добавления информации о жанре при выдаче книги на
чтение.
* Научная литература (***ScientificLiterature***) (если требуется третий класс):
Должен содержать дополнительные атрибуты, такие как Область науки
(***FieldOfScience***). Метод *GetInfo()* должен быть переопределен для включения
информации об области науки в описании книги.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [5]:
// Интерфейс, который определяет методы для выдачи (Borrow) и возврата (Return) книги.
public interface IBorrowable
{
    void Borrow();
    void Borrow(int days);
    void Return();
}

// Интерфейс для сервиса уведомлений
public interface INotificationService
{
    void SendNotification(string message);
    event Action<string> NotificationSent;
}

// Интерфейс репозитория для работы с книгами
public interface IBookRepository
{
    void UpdateBookStatus(string isbn, bool isAvailable);
    Book GetBookByIsbn(string isbn);
    void AddBook(Book book);
    IEnumerable<Book> GetAllBooks();
    event Action<Book> BookAdded;
    event Action<string> BookStatusChanged;
}

// Делегат для обработки событий книг
public delegate void BookEventHandler(Book book, string action);

// Базовый класс Book с явной реализацией интерфейса IBorrowable
public class Book : IBorrowable
{
    public string Title { get; set; }
    public string Author { get; set; }
    public int YearOfPublication { get; set; }
    public string ISBN { get; set; }
    public string Publisher { get; set; }
    public int NumberOfPages { get; set; }
    public string Language { get; set; }
    public bool IsAvailable { get; internal set; } = true;

    // Событие, возникающее при изменении статуса книги
    public event Action<Book, bool> AvailabilityChanged;

    private static int bookCount = 0;
    private readonly INotificationService _notificationService;
    private readonly IBookRepository _bookRepository;

    public Book(string title, string author, int yearOfPublication, string isbn, 
               string publisher, int numberOfPages, string language,
               INotificationService notificationService, IBookRepository bookRepository)
    {
        Title = title ?? throw new ArgumentNullException(nameof(title));
        Author = author ?? throw new ArgumentNullException(nameof(author));
        YearOfPublication = yearOfPublication;
        ISBN = isbn ?? throw new ArgumentNullException(nameof(isbn));
        Publisher = publisher ?? throw new ArgumentNullException(nameof(publisher));
        NumberOfPages = numberOfPages;
        Language = language ?? throw new ArgumentNullException(nameof(language));
        _notificationService = notificationService ?? throw new ArgumentNullException(nameof(notificationService));
        _bookRepository = bookRepository ?? throw new ArgumentNullException(nameof(bookRepository));
        bookCount++;
        
        _bookRepository.AddBook(this);
        
        // Подписка на события репозитория
        _bookRepository.BookAdded += OnBookAddedToRepository;
        _bookRepository.BookStatusChanged += OnBookStatusChangedInRepository;
    }

    private void OnBookAddedToRepository(Book book)
    {
        if (book == this)
        {
            _notificationService.SendNotification($"Книга '{Title}' добавлена в репозиторий.");
        }
    }

    private void OnBookStatusChangedInRepository(string isbn)
    {
        if (isbn == ISBN)
        {
            _notificationService.SendNotification($"Статус книги '{Title}' был изменен в репозитории.");
        }
    }

    public static int GetBookCount() => bookCount;

    public virtual string GetInfo()
    {
        return $"Название: {Title}, Автор: {Author}, Год: {YearOfPublication}, ISBN: {ISBN}, " +
               $"\nИздательство: {Publisher}, Страниц: {NumberOfPages}, Язык: {Language}, " +
               $"\nДоступна: {(IsAvailable ? "Да" : "Нет")}";
    }

    public virtual void Read()
    {
        Console.WriteLine($"Чтение книги: {Title} ({Author}, {YearOfPublication})");
    }

    public void CheckAvailability()
    {
        Console.WriteLine($"Книга '{Title}' доступна: {(IsAvailable ? "Да" : "Нет")}.");
    }

    // Явная реализация интерфейса IBorrowable
    void IBorrowable.Borrow() => ProcessBorrow($"Книга '{Title}' выдана на чтение.");

    void IBorrowable.Borrow(int days) => ProcessBorrow($"Книга '{Title}' выдана на {days} дней.");

    void IBorrowable.Return()
    {
        IsAvailable = true;
        _bookRepository.UpdateBookStatus(ISBN, true);
        _notificationService.SendNotification($"Книга '{Title}' возвращена.");
        AvailabilityChanged?.Invoke(this, true);
    }

    private void ProcessBorrow(string successMessage)
    {
        if (IsAvailable)
        {
            IsAvailable = false;
            _bookRepository.UpdateBookStatus(ISBN, false);
            _notificationService.SendNotification(successMessage);
            AvailabilityChanged?.Invoke(this, false);
        }
        else
        {
            _notificationService.SendNotification($"Книга '{Title}' недоступна для выдачи.");
        }
    }
}

// Класс учебника
public class Textbook : Book
{
    public string Subject { get; set; }
    public int Edition { get; set; }
    public bool IsRecommended { get; set; }

    public Textbook(string title, string author, int yearOfPublication, string isbn, 
                   string publisher, int numberOfPages, string language,
                   INotificationService notificationService, IBookRepository bookRepository,
                   string subject, int edition, bool isRecommended)
        : base(title, author, yearOfPublication, isbn, publisher, numberOfPages, 
              language, notificationService, bookRepository)
    {
        Subject = subject ?? throw new ArgumentNullException(nameof(subject));
        Edition = edition;
        IsRecommended = isRecommended;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nПредмет: {Subject}, Издание: {Edition}, " +
               $"Рекомендован: {(IsRecommended ? "Да" : "Нет")}";
    }

    public override void Read()
    {
        Console.WriteLine($"Чтение учебника по {Subject}: {Title} (Изд. {Edition}).");
    }

    public void CheckRecommendation()
    {
        Console.WriteLine($"Учебник '{Title}' рекомендован: {(IsRecommended ? "Да" : "Нет")}.");
    }
}

// Класс художественной литературы
public class Fiction : Book
{
    public string Genre { get; set; }
    public string Awards { get; set; }
    public bool IsBestseller { get; set; }

    public Fiction(string title, string author, int yearOfPublication, string isbn,
                  string publisher, int numberOfPages, string language,
                  INotificationService notificationService, IBookRepository bookRepository,
                  string genre, string awards, bool isBestseller)
        : base(title, author, yearOfPublication, isbn, publisher, numberOfPages,
              language, notificationService, bookRepository)
    {
        Genre = genre ?? throw new ArgumentNullException(nameof(genre));
        Awards = awards;
        IsBestseller = isBestseller;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nЖанр: {Genre}, Награды: {Awards}, " +
               $"Бестселлер: {(IsBestseller ? "Да" : "Нет")}";
    }

    public override void Read()
    {
        base.Read();
        Console.WriteLine($"Погружение в мир {Genre}...");
    }

    public void Read(string readingMode)
    {
        Console.WriteLine($"Чтение '{Title}' в режиме {readingMode}.");
    }

    public void CheckBestsellerStatus()
    {
        Console.WriteLine($"Книга '{Title}' - бестселлер: {(IsBestseller ? "Да" : "Нет")}.");
    }
}

// Класс научной литературы
public class ScientificLiterature : Book
{
    public string FieldOfScience { get; set; }
    public int Citations { get; set; }
    public bool IsPeerReviewed { get; set; }

    public ScientificLiterature(string title, string author, int yearOfPublication, string isbn,
                               string publisher, int numberOfPages, string language,
                               INotificationService notificationService, IBookRepository bookRepository,
                               string fieldOfScience, int citations, bool isPeerReviewed)
        : base(title, author, yearOfPublication, isbn, publisher, numberOfPages,
              language, notificationService, bookRepository)
    {
        FieldOfScience = fieldOfScience ?? throw new ArgumentNullException(nameof(fieldOfScience));
        Citations = citations;
        IsPeerReviewed = isPeerReviewed;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nОбласть: {FieldOfScience}, Цитирований: {Citations}, " +
               $"Рецензирована: {(IsPeerReviewed ? "Да" : "Нет")}.";
    }

    public void CheckPeerReviewStatus()
    {
        Console.WriteLine($"Научная книга '{Title}' рецензирована: {(IsPeerReviewed ? "Да" : "Нет")}.");
    }

    public void Return(string comments)
    {
        ((IBorrowable)this).Return();
        Console.WriteLine($"Комментарий при возврате: '{comments}'");
    }
}

// Продвинутый учебник
public class AdvancedTextbook : Textbook
{
    public string DifficultyLevel { get; set; }
    public bool IsDigital { get; set; }

    public AdvancedTextbook(string title, string author, int yearOfPublication, string isbn,
                           string publisher, int numberOfPages, string language,
                           INotificationService notificationService, IBookRepository bookRepository,
                           string subject, int edition, bool isRecommended,
                           string difficultyLevel, bool isDigital)
        : base(title, author, yearOfPublication, isbn, publisher, numberOfPages,
              language, notificationService, bookRepository, subject, edition, isRecommended)
    {
        DifficultyLevel = difficultyLevel ?? throw new ArgumentNullException(nameof(difficultyLevel));
        IsDigital = isDigital;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nУровень: {DifficultyLevel}, " +
               $"Цифровая: {(IsDigital ? "Да" : "Нет")}";
    }

    public override void Read()
    {
        Console.WriteLine($"Изучение продвинутого учебника '{Title}' (Уровень: {DifficultyLevel}).");
    }

    public void CheckDigitalAvailability()
    {
        Console.WriteLine($"Цифровая версия '{Title}': {(IsDigital ? "Есть" : "Нет")}.");
    }

    public void CheckDigitalAvailability(bool showPrice)
    {
        CheckDigitalAvailability();
        if (showPrice && IsDigital)
        {
            Console.WriteLine("Цена цифровой версии: 300₽");
        }
    }
}

// Историческая художественная литература
public class HistoricalFiction : Fiction
{
    public string HistoricalPeriod { get; set; }
    public string HistoricalAccuracy { get; set; }
    public string MainCharacter { get; set; }

    public HistoricalFiction(string title, string author, int yearOfPublication, string isbn,
                            string publisher, int numberOfPages, string language,
                            INotificationService notificationService, IBookRepository bookRepository,
                            string genre, string awards, bool isBestseller,
                            string historicalPeriod, string historicalAccuracy, string mainCharacter)
        : base(title, author, yearOfPublication, isbn, publisher, numberOfPages,
              language, notificationService, bookRepository, genre, awards, isBestseller)
    {
        HistoricalPeriod = historicalPeriod ?? throw new ArgumentNullException(nameof(historicalPeriod));
        HistoricalAccuracy = historicalAccuracy;
        MainCharacter = mainCharacter;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nПериод: {HistoricalPeriod}, " +
               $"Точность: {HistoricalAccuracy}, Герой: {MainCharacter}";
    }

    public void CheckHistoricalAccuracy()
    {
        Console.WriteLine($"Историческая точность '{Title}': {HistoricalAccuracy}.");
    }

    public void CheckBestsellerStatus(string region)
    {
        Console.WriteLine($"Бестселлер в {region}: {(IsBestseller ? "Да" : "Нет")}.");
    }
}

// Реализация репозитория книг
public class BookRepository : IBookRepository
{
    private readonly Dictionary<string, Book> _books = new Dictionary<string, Book>();
    
    public event Action<Book> BookAdded;
    public event Action<string> BookStatusChanged;

    public void UpdateBookStatus(string isbn, bool isAvailable)
    {
        if (_books.TryGetValue(isbn, out var book))
        {
            book.IsAvailable = isAvailable;
            BookStatusChanged?.Invoke(isbn);
        }
    }

    public Book GetBookByIsbn(string isbn) => _books.TryGetValue(isbn, out var book) ? book : null;
    
    public void AddBook(Book book)
    {
        if (book == null) throw new ArgumentNullException(nameof(book));
        _books[book.ISBN] = book;
        BookAdded?.Invoke(book);
    }
    
    public IEnumerable<Book> GetAllBooks() => _books.Values;
}

// Реализация сервиса уведомлений
public class EmailNotificationService : INotificationService
{
    public event Action<string> NotificationSent;

    public void SendNotification(string message)
    {
        Console.WriteLine($"[Email] {DateTime.Now}: {message}");
        NotificationSent?.Invoke(message);
    }
}

// Generic класс для управления коллекцией книг
public class Library<T> where T : Book
{
    private readonly List<T> _books = new List<T>();
    private readonly Dictionary<string, T> _booksByIsbn = new Dictionary<string, T>();
    
    public event Action<T> BookAdded;
    public event Action<T> BookRemoved;

    public void AddBook(T book)
    {
        if (book == null) throw new ArgumentNullException(nameof(book));
        
        if (!_booksByIsbn.ContainsKey(book.ISBN))
        {
            _books.Add(book);
            _booksByIsbn.Add(book.ISBN, book);
            Console.WriteLine($"Добавлена: '{book.Title}'");
            BookAdded?.Invoke(book);
            
            // Подписка на событие изменения доступности книги
            book.AvailabilityChanged += OnBookAvailabilityChanged;
        }
    }

    private void OnBookAvailabilityChanged(Book book, bool isAvailable)
    {
        Console.WriteLine($"В библиотеке: книга '{book.Title}' теперь {(isAvailable ? "доступна" : "недоступна")}");
    }

    public bool RemoveBook(T book)
    {
        if (book == null) return false;
        
        if (_books.Remove(book))
        {
            _booksByIsbn.Remove(book.ISBN);
            BookRemoved?.Invoke(book);
            return true;
        }
        return false;
    }

    public T FindBookByTitle(string title) => 
        _books.FirstOrDefault(b => b.Title.Equals(title, StringComparison.OrdinalIgnoreCase));

    public T FindBookByIsbn(string isbn) => 
        _booksByIsbn.TryGetValue(isbn, out var book) ? book : null;

    public void DisplayAllBooks()
    {
        Console.WriteLine($"\nВ библиотеке {_books.Count} книг:");
        foreach (var book in _books.OrderBy(b => b.Title))
        {
            Console.WriteLine(book.GetInfo());
        }
    }
    
    public IEnumerable<T> GetAvailableBooks() => _books.Where(b => b.IsAvailable);
    public IEnumerable<T> GetBorrowedBooks() => _books.Where(b => !b.IsAvailable);
}

In [6]:
// Инициализация зависимостей
var notificationService = new EmailNotificationService();
var bookRepository = new BookRepository();

// Подписка на события сервиса уведомлений
notificationService.NotificationSent += message => 
    Console.WriteLine($"[Лог] Уведомление отправлено: {message}");

// Подписка на события репозитория
bookRepository.BookAdded += book => 
    Console.WriteLine($"[Лог] Книга добавлена в репозиторий: {book.Title}");
bookRepository.BookStatusChanged += isbn => 
    Console.WriteLine($"[Лог] Статус книги с ISBN {isbn} был изменен");

Console.WriteLine("=== Создание объектов всех типов книг ===\n");

// 1. Создание обычной книги
var book = new Book(
    "CLR via C#", "Джеффри Рихтер", 2018,
    "978-5-4461-1045-3", "Питер", 896, "Русский",
    notificationService, bookRepository);

Console.WriteLine($"Создана книга: {book.GetInfo()}\n");

// 2. Создание учебника
var textbook = new Textbook(
    "Изучаем Python", "Марк Лутц", 2019,
    "978-5-6041394-7-3", "Диалектика", 1648, "Русский",
    notificationService, bookRepository,
    "Программирование", 5, true);

Console.WriteLine($"Создан учебник: {textbook.GetInfo()}\n");

// 3. Создание художественной литературы
var fiction = new Fiction(
    "451° по Фаренгейту", "Рэй Брэдбери", 1953,
    "978-5-17-090876-5", "АСТ", 256, "Русский",
    notificationService, bookRepository,
    "Антиутопия", "Премия Хьюго", true);

Console.WriteLine($"Создана художественная книга: {fiction.GetInfo()}\n");

// 4. Создание научной литературы
var scienceBook = new ScientificLiterature(
    "Краткая история времени", "Стивен Хокинг", 1988,
    "978-5-17-090879-6", "АСТ", 220, "Русский",
    notificationService, bookRepository,
    "Космология", 25000, true);

Console.WriteLine($"Создана научная книга: {scienceBook.GetInfo()}\n");

// 5. Создание продвинутого учебника
var advancedTextbook = new AdvancedTextbook(
    "Паттерны проектирования", "Эрик Фримен", 2021,
    "978-5-4461-1456-7", "Питер", 656, "Русский",
    notificationService, bookRepository,
    "Программирование", 2, true,
    "Профессиональный", true);

Console.WriteLine($"Создан продвинутый учебник: {advancedTextbook.GetInfo()}\n");

// 6. Создание исторической художественной литературы
var historicalFiction = new HistoricalFiction(
    "Война и мир", "Лев Толстой", 1869,
    "978-5-389-06256-6", "Азбука", 1360, "Русский",
    notificationService, bookRepository,
    "Роман-эпопея", "Литературная премия", true,
    "Эпоха Наполеона", "Высокая", "Пьер Безухов");

Console.WriteLine($"Создана историческая книга: {historicalFiction.GetInfo()}\n");

Console.WriteLine("=== Демонстрация работы с книгами ===\n");

// Работа через интерфейс IBorrowable
Console.WriteLine("--- Выдача и возврат книг ---");
IBorrowable[] allBooks = { book, textbook, fiction, scienceBook, advancedTextbook, historicalFiction };

foreach (var item in allBooks)
{
    Console.WriteLine($"\nРаботаем с: {item.GetType().Name}");
    item.Borrow();
    item.Borrow(14); // перегруженный метод
    item.Return();
}

Console.WriteLine($"\nВозьмем одну книгу на чтение (без возврата):");
allBooks[0].Borrow();

// Использование специализированных методов
Console.WriteLine("\n--- Специализированные методы ---");
textbook.CheckRecommendation();
fiction.CheckBestsellerStatus();
scienceBook.CheckPeerReviewStatus();
advancedTextbook.CheckDigitalAvailability(true);
historicalFiction.CheckHistoricalAccuracy();
historicalFiction.CheckBestsellerStatus("России");

// Работа с библиотекой
Console.WriteLine("\n--- Работа с библиотекой ---");
var library = new Library<Book>();

// Подписка на события библиотеки
library.BookAdded += addedBook => 
    Console.WriteLine($"[Библиотека] Добавлена новая книга: {addedBook.Title}");
library.BookRemoved += removedBook => 
    Console.WriteLine($"[Библиотека] Книга удалена: {removedBook.Title}");

library.AddBook(book);
library.AddBook(textbook);
library.AddBook(fiction);
library.AddBook(scienceBook);
library.AddBook(advancedTextbook);
library.AddBook(historicalFiction);

library.DisplayAllBooks();

// Поиск книги
Console.WriteLine("\n--- Поиск книги ---");
var foundBook = library.FindBookByTitle("451° по Фаренгейту");
if (foundBook != null)
{
    Console.WriteLine($"Найдена книга: {foundBook.GetInfo()}");
    foundBook.Read();
}

// Работа с коллекциями
Console.WriteLine("\n--- Доступные книги ---");
foreach (var availableBook in library.GetAvailableBooks())
{
    Console.WriteLine($"Доступна: {availableBook.Title}");
}

Console.WriteLine("\n--- Выданные книги ---");
foreach (var borrowedBook in library.GetBorrowedBooks())
{
    Console.WriteLine($"Выдана: {borrowedBook.Title}");
}

Console.WriteLine("\n=== Статистика ===");
Console.WriteLine($"Всего создано книг: {Book.GetBookCount()}");
Console.WriteLine($"Книг в библиотеке: {library.GetAvailableBooks().Count() + library.GetBorrowedBooks().Count()}");

=== Создание объектов всех типов книг ===

[Лог] Книга добавлена в репозиторий: CLR via C#
Создана книга: Название: CLR via C#, Автор: Джеффри Рихтер, Год: 2018, ISBN: 978-5-4461-1045-3, 
Издательство: Питер, Страниц: 896, Язык: Русский, 
Доступна: Да

[Лог] Книга добавлена в репозиторий: Изучаем Python
Создан учебник: Название: Изучаем Python, Автор: Марк Лутц, Год: 2019, ISBN: 978-5-6041394-7-3, 
Издательство: Диалектика, Страниц: 1648, Язык: Русский, 
Доступна: Да
Предмет: Программирование, Издание: 5, Рекомендован: Да

[Лог] Книга добавлена в репозиторий: 451° по Фаренгейту
Создана художественная книга: Название: 451° по Фаренгейту, Автор: Рэй Брэдбери, Год: 1953, ISBN: 978-5-17-090876-5, 
Издательство: АСТ, Страниц: 256, Язык: Русский, 
Доступна: Да
Жанр: Антиутопия, Награды: Премия Хьюго, Бестселлер: Да

[Лог] Книга добавлена в репозиторий: Краткая история времени
Создана научная книга: Название: Краткая история времени, Автор: Стивен Хокинг, Год: 1988, ISBN: 978-5-17-090879-6, 
